
*Note: the functions in this notebook were written with the help of a coding agent, based on the process I designed and tested for comp1 in `02_initial_precursors.ipynb`. I reviewed the code and confirmed it reproduces the comp1 results. Final Analysis and Findings were completed by me*

# Precursors for All Components
This notebook repeats the process from `02_initial_precursors.ipynb` (where comp1 was worked through in detail) for **all four components**. 

The three candidates that worked for comp1 are carried forward:
1. **Matching error**
2. **Sustained sensor shift**
3. **Error + sensor**

The fourth comp1 candidate (3+ consecutive raw outliers) was rejected in `02_initial_precursors.ipynb`, so it isn't repeated here.

All design choices (events, H and W, normal time, thresholds, metrics) are the same as in `02_initial_precursors.ipynb`. As a check, the comp1 results here match that notebook exactly.

The EDA showed each component failure lines up with its own error type(s) and one telemetry sensor:

- **comp1:** failures follow **error1** and a **rise in voltage**.
- **comp2:** failures follow **error2 / error3** and a **drop in rotate**.
- **comp3:** failures follow **error4** and a **rise in pressure**.
- **comp4:** failures follow **error5** and a **rise in vibration**.

# Imports

In [1]:
import kagglehub
import pandas as pd

/Users/jaredpaul/Coding/Predictive-Maintenance-RCA/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/jaredpaul/Coding/Predictive-Maintenance-RCA/.venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Load Data

In [2]:
path = kagglehub.dataset_download("arnabbiswas1/microsoft-azure-predictive-maintenance")

telemetry_df = pd.read_csv(f"{path}/PdM_telemetry.csv", parse_dates=["datetime"])
errors_df = pd.read_csv(f"{path}/PdM_errors.csv", parse_dates=["datetime"])
failures_df = pd.read_csv(f"{path}/PdM_failures.csv", parse_dates=["datetime"])
machines_df = pd.read_csv(f"{path}/PdM_machines.csv")

# Settings
H and W come from the EDA (errors ~24h and sensor shifts ~48h before failure). `COMP_SETUP` holds each component's matching error(s), sensor, and the direction the sensor shifts before failure.

In [3]:
H = 12   # hours of warning before the failure
W = 36   # length of the warning window in hours

# Each component's matching error(s), sensor, and which direction the sensor shifts before failure (from EDA)
COMP_SETUP = {
    "comp1": {"errors": ["error1"],           "sensor": "volt",      "direction": "high"},
    "comp2": {"errors": ["error2", "error3"], "sensor": "rotate",    "direction": "low"},
    "comp3": {"errors": ["error4"],           "sensor": "pressure",  "direction": "high"},
    "comp4": {"errors": ["error5"],           "sensor": "vibration", "direction": "high"},
}

# Metric Functions
Same as `02_precursors.ipynb`, but they take the grid (and events) as inputs so they work for any component. `count_alerts` counts how many times an alert switches from off to on.

In [4]:
def calc_lift(grid_df, alert_col):
    rate_in_window = grid_df.loc[grid_df["in_window"], alert_col].mean()
    rate_in_normal = grid_df.loc[grid_df["normal"], alert_col].mean()
    return rate_in_window / rate_in_normal


def calc_precision(grid_df, alert_col):
    # An alert = the moment it switches from off to on
    was_on_last_hour = grid_df.groupby("machineID")[alert_col].shift(1, fill_value=False)
    alert_starts_df = grid_df[grid_df[alert_col] & ~was_on_last_hour]

    # Correct alerts = ones that start inside a warning window
    return alert_starts_df["in_window"].mean()


def calc_coverage(grid_df, events_df, alert_col):
    # Failures whose warning window had at least one alert
    caught = grid_df[grid_df[alert_col] & grid_df["in_window"]]["window_id"].nunique()
    return caught / len(events_df)


def count_alerts(grid_df, alert_col):
    # Number of times the alert switches from off to on
    was_on_last_hour = grid_df.groupby("machineID")[alert_col].shift(1, fill_value=False)
    return int((grid_df[alert_col] & ~was_on_last_hour).sum())

# Build the Grid for a Component
Events, warning window (`in_window`) and normal time (`normal`) for one component.

In [5]:
def build_grid(comp):
    # Events = failures of this component, skipping the first 48h of data (no history before them)
    data_start = telemetry_df["datetime"].min()
    events_df = failures_df[
        (failures_df["failure"] == comp) &
        (failures_df["datetime"] >= data_start + pd.Timedelta(hours=48))
    ]

    # Hourly grid with this component's sensor
    sensor = COMP_SETUP[comp]["sensor"]
    grid_df = telemetry_df[["machineID", "datetime", sensor]].copy()

    # Warning window: 48h to 12h before each failure of this component
    grid_df["window_id"] = None
    for failure_id, failure in events_df.iterrows():
        grid_df.loc[
            (grid_df["machineID"] == failure["machineID"]) &
            (grid_df["datetime"] >= failure["datetime"] - pd.Timedelta(hours=H + W)) &
            (grid_df["datetime"] < failure["datetime"] - pd.Timedelta(hours=H)),
            "window_id"
        ] = failure_id
    grid_df["in_window"] = grid_df["window_id"].notna()

    # Normal time: not within H + W before any failure (of any component)
    grid_df["normal"] = True
    for _, failure in failures_df.iterrows():
        grid_df.loc[
            (grid_df["machineID"] == failure["machineID"]) &
            (grid_df["datetime"] >= failure["datetime"] - pd.Timedelta(hours=H + W)) &
            (grid_df["datetime"] <= failure["datetime"]),
            "normal"
        ] = False

    return grid_df, events_df

# Add the Detectors
1. **Matching error:** any of the component's errors is logged.
2. **Sustained sensor shift:** 12h average past the 99th percentile of normal (1st percentile for rotate, which drops), held on for 12h to stop flickering.
3. **Error + sensor:** an error is logged while the sensor alert is on.

In [6]:
def add_detectors(grid_df, comp):
    setup = COMP_SETUP[comp]
    sensor = setup["sensor"]

    # Candidate 1: matching error alert (any of the component's errors)
    comp_errors_df = errors_df[errors_df["errorID"].isin(setup["errors"])][["machineID", "datetime"]].drop_duplicates()
    comp_errors_df["error_alert"] = True
    grid_df = grid_df.merge(comp_errors_df, on=["machineID", "datetime"], how="left")
    grid_df["error_alert"] = grid_df["error_alert"].fillna(False).astype(bool)

    # Candidate 2: sustained sensor shift (12h average past its 99th / 1st percentile of normal)
    grid_df["sensor_12h"] = grid_df.groupby("machineID")[sensor].transform(lambda x: x.rolling(12).mean())
    normal_values = grid_df.loc[grid_df["normal"], "sensor_12h"]

    if setup["direction"] == "high":
        threshold = normal_values.quantile(0.99)
        grid_df["sensor_alert"] = grid_df["sensor_12h"] > threshold
    else:
        threshold = normal_values.quantile(0.01)
        grid_df["sensor_alert"] = grid_df["sensor_12h"] < threshold

    # Keep the alert on until the sensor has been back past the threshold for 12h (stops flickering)
    grid_df["sensor_alert_steady"] = (
        grid_df.groupby("machineID")["sensor_alert"]
        .transform(lambda x: x.rolling(12, min_periods=1).max())
        .astype(bool)
    )

    # Candidate 3: error alert while the sensor alert is on
    grid_df["combined_alert"] = grid_df["error_alert"] & grid_df["sensor_alert_steady"]

    return grid_df

# Run One Component
Builds the grid, adds the detectors and returns the number of failures and alerts, plus lift, precision and coverage for each detector.

In [7]:
def run_component(comp):
    grid_df, events_df = build_grid(comp)
    grid_df = add_detectors(grid_df, comp)

    detectors = {
        "error_alert": "Matching error",
        "sensor_alert_steady": "Sustained sensor shift",
        "combined_alert": "Error + sensor",
    }

    results = []
    for alert_col, name in detectors.items():
        results.append({
            "component": comp,
            "detector": name,
            "failures": len(events_df),
            "alerts": count_alerts(grid_df, alert_col),
            "lift": round(calc_lift(grid_df, alert_col), 1),
            "precision": round(calc_precision(grid_df, alert_col), 2),
            "coverage": round(calc_coverage(grid_df, events_df, alert_col), 2),
        })
    return results

# Results for All Components

In [8]:
all_results = []
for comp in COMP_SETUP:
    all_results += run_component(comp)

results_df = pd.DataFrame(all_results)
results_df

/var/folders/h3/58sb5ybx6fzgy8l9n1029qd80000gn/T/ipykernel_29650/1537976066.py:9: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  grid_df["error_alert"] = grid_df["error_alert"].fillna(False).astype(bool)
/var/folders/h3/58sb5ybx6fzgy8l9n1029qd80000gn/T/ipykernel_29650/1537976066.py:9: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  grid_df["error_alert"] = grid_df["error_alert"].fillna(False).astype(bool)
/var/folders/h3/58sb5ybx6fzgy8l9n1029qd80000gn/T/ipykernel_29650/1537976066.py:9: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfil

,component,detector,failures,alerts,lift,precision,coverage
0,comp1,Matching error,181,1010,30.4,0.19,1.00
1,comp1,Sustained sensor shift,181,867,23.3,0.21,0.91
2,comp1,Error + sensor,181,150,1208.3,0.87,0.71
3,comp2,Matching error,256,1569,19.6,0.17,1.00
4,comp2,Sustained sensor shift,256,961,22.6,0.26,0.93
5,comp2,Error + sensor,256,238,603.8,0.86,0.77
6,comp3,Matching error,127,727,41.5,0.18,1.00
7,comp3,Sustained sensor shift,127,807,26.0,0.15,0.95
8,comp3,Error + sensor,127,123,1702.4,0.90,0.86
9,comp4,Matching error,176,356,137.4,0.50,1.00


# Final Analysis and Findings

- The same Pattern holds for every component where the error precursor and telemetry sensor precursor has higher coverage but most of the alerts are false alarms with very low precision.

- The Precursors combining both error and sensor have not as much coverage but significantly better precision.

- **Ultimately, I believe that tradeoff in lower coverage but high precision makes the Error+Sensor Precursors the best choice because bombarding technicians with false alarms can quickly lead to frustration and a loss of trust in the alert system. If too many alerts turn out to be nothing, workers may start ignoring them altogether, including the ones that matter.**

- The **3 Precursors** I would keep are the **Error+Sensor Precursors** for **component 2, component 3, and component 4.**
    - Component 3 and 4 precursors clearly have the best performance with the highest precision and coverage.
    - Component 2 and 1 precursors are similar but comp2 catches more failures and has the most failures overall. 

## Shortlist

1. **comp2 failure:** error2 or error3 is logged while rotate's 12h average is below its 1st percentile of normal operation.
2. **comp3 failure:** error4 is logged while pressure's 12h average is above its 99th percentile of normal operation.
3. **comp4 failure:** error5 is logged while vibration's 12h average is above its 99th percentile of normal operation.

## With More Time

- **Stories for comp2–4:** I wrote a story and detector for each comp1 candidate in notebook 02. Skipped over writing the story for the other component precursors given that they followed the same design style. With more time, I would have made sure to write the story for each one.

- **Manually validating comp2–4:** I developed and tested the precursor design and test on comp1, then used my coding agent to apply it to the remaining components. With more time, I would work through comp2–4 individually to validate the candidate detectors and refine the approach for each component, as I did with comp1.

- **Testing other thresholds:** I would test out more percentiles and rolling windows other than the 99th percentile and 12h mean.

- **Unused EDA findings:** a few things from the EDA could make the detectors sharper: component age (failures almost never happen in a part's first ~30 days), machine age (older machines fail more), machine model (comp3 never fails on models 3 and 4), and time since the last failure. I'd test these as filters on the existing detectors.

- **In hindsight**, I probably allocated too much time on exploring the data, which left me with less time for testing and experimenting with my precursors.